In [1]:
import os
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from scipy.spatial import cKDTree

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups
from lvt.philadelphia import tax_year_params, parcel_cache_path

CITY_NAME = 'philadelphia'
STATE_FIPS = '42'
COUNTY_FIPS = '101'
LAND_IMPROVEMENT_RATIO = 4.0

# GMA zone assignment: static reference extracted from OPA's 2025 GMA PDF
# (parcel centroid → L1/L2/L3 zone labels; 17 / 84 / 613 zones)
GMA_PATH = Path('data/parcel_gma_assignment.parquet')

# --- Tax year -------------------------------------------------------------------
# Rates and the revenue-validation target live in lvt/philadelphia.py, keyed by year and
# cited there. Do NOT hardcode a millage here: the combined rate has been 1.3998% for
# years, but the City/School split moved at TY2025, which silently invalidates the
# city-only cross-check without changing anything the model computes.
TAX_YEAR = int(os.environ.get('LVT_TAX_YEAR', 2026))   # override: LVT_TAX_YEAR=2027
TY = tax_year_params(TAX_YEAR)
MILLAGE = TY.combined_mills
PARCEL_PATH = parcel_cache_path(TAX_YEAR)
MODEL_TYPE = f'split_rate_4to1_lycd_post_abatement_ty{TAX_YEAR}'
EXPORT_SUFFIX = f'_lycd_post_abatement_ty{TAX_YEAR}'

print(TY.describe())

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)


TY2026: 0.6159% city + 0.7839% school = 1.3998% (13.998 mills) | city target $891,102,000 (projection)


C:\Users\druss\miniconda3\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Step 1: Load parcel data

In [2]:
if not PARCEL_PATH.exists():
    raise FileNotFoundError(
        f'{PARCEL_PATH} not found. Build it with:\n'
        f'    python scripts/build_philadelphia_parcel_cache.py --year {TAX_YEAR}\n'
        'The cache is keyed by tax year on purpose — opa_properties_public always carries '
        'the latest assessment year, so an unsuffixed cache makes it easy to model one '
        "year's taxable values against another year's expectations with no visible symptom."
    )
gdf = gpd.read_parquet(PARCEL_PATH)
_required = {'parcel_number', 'taxable_land', 'taxable_building', 'market_value',
             'exempt_land', 'exempt_building', 'pin', 'category_code', 'total_area'}
_missing = _required - set(gdf.columns)
if _missing:
    raise ValueError(
        f'{PARCEL_PATH} is missing columns {sorted(_missing)} — rebuild with '
        f'scripts/build_philadelphia_parcel_cache.py --year {TAX_YEAR} --force'
    )
gdf['parcel_number'] = gdf['parcel_number'].astype(str).str.zfill(9)
print(f'Loaded {len(gdf):,} parcels for TY{TAX_YEAR}')
print(f'  taxable base: ${(gdf["taxable_land"].sum() + gdf["taxable_building"].sum())/1e9:.3f}B')


Loaded 583,204 parcels for TY2026
  taxable base: $153.068B


## Step 2: Lot area — one convention, no spatial join

LYCD land value is `zone_psf × lot_area`, and the zone PSF is itself `median(market_value /
lot_area)`. That makes the method **exactly scale-invariant**: multiplying every parcel's area
by a constant changes no result. What it is *not* invariant to is mixing two area conventions,
because that inflates one group's land value relative to everyone else's.

An earlier version of this notebook did exactly that. It sourced ~94% of parcels from OPA
`total_area` (true ground area) and fell back for ~5% to a spatial join against DOR polygons
whose `Shape__Area` is computed in **EPSG:3857**. Web Mercator inflates area by
`1/cos²(latitude)` ≈ **1.704×** at Philadelphia's latitude, so those parcels arrived on an area
scale 1.7× larger than the rest — and, because the join was point-in-polygon, every parcel
inside one large polygon inherited its *full* area. The two effects together pushed the total
lot area to 3.08× the city's actual land area and drove ~62% of what the market-value cap in
Step 4 has to clip.

The chain below fixes both:

1. **OPA `total_area`** — true ground area, one-to-one, covers ~94.5%.
2. **DOR PIN-keyed area** from `parcel_areas_by_pin_current.parquet`, fetched by
   `scripts/fetch_dor_parcel_areas.py` and **de-distorted** to true ground sqft with a
   per-parcel `cos²(latitude)` correction. Used where OPA has no area, and to override OPA
   where the two disagree by more than 3× (a handful of bad OPA records).
3. **KNN (5 nearest neighbours)** for the remainder, imputing from parcels already on the
   same convention.

No spatial join. After the fix the corrected PIN areas agree with OPA `total_area` at a median
ratio of 0.995, and total lot area lands at ~0.80× the city — plausible, since parcels exclude
streets, rights-of-way and water.

In [3]:
PIN_AREA_PATH = DATA_DIR / 'parcel_areas_by_pin_current.parquet'

if not PIN_AREA_PATH.exists():
    raise FileNotFoundError(
        f'{PIN_AREA_PATH} not found. Build it with:\n'
        '    python scripts/fetch_dor_parcel_areas.py\n'
        'Do NOT fall back to parcel_areas_by_pin.parquet — those areas are Web Mercator '
        '(inflated ~1.704x at this latitude) and mixing them with OPA total_area puts ~5% of '
        'parcels on a different area scale. See the Step 2 notes above.'
    )

pin_areas = pd.read_parquet(PIN_AREA_PATH)
pin_areas['pin'] = pin_areas['pin'].astype(str).str.strip()
print(f'PIN-area lookup: {len(pin_areas):,} PINs (true ground sqft, Mercator-corrected)')
print(f'  median lot: {pin_areas["pin_area_sqft"].median():,.0f} sqft')

PIN-area lookup: 580,097 PINs (true ground sqft, Mercator-corrected)
  median lot: 1,365 sqft


## Step 3: Load GMA zone assignments and join lot area to parcels

In [4]:
gma = pd.read_parquet(GMA_PATH)
gma['key'] = gma['key'].astype(str).str.zfill(9)
print(f'GMA assignment: {len(gma):,} parcels | '
      f'L1={gma["gma1"].nunique()} zones | L2={gma["gma2"].nunique()} | L3={gma["gma3"].nunique()}')

# --- Lot area: single convention (true ground sqft), no spatial join ---
OPA_PIN_DISAGREE_FACTOR = 3.0   # above this, trust the surveyed polygon over OPA's record

gdf['pin'] = gdf['pin'].astype(str).str.strip()
_opa_area = pd.to_numeric(gdf['total_area'], errors='coerce').fillna(0)
_pin_area = gdf['pin'].map(pin_areas.drop_duplicates('pin').set_index('pin')['pin_area_sqft'])

_has_opa = _opa_area > 0
_has_pin = _pin_area.notna() & (_pin_area > 0)
# A few OPA records are implausible (e.g. a single parcel recorded at 7.45 sq mi).
# Where a surveyed DOR polygon disagrees by more than the factor above, prefer the polygon.
_override = _has_opa & _has_pin & ((_opa_area / _pin_area) > OPA_PIN_DISAGREE_FACTOR)

_area = pd.Series(np.where(_override, _pin_area,
                  np.where(_has_opa, _opa_area,
                  np.where(_has_pin, _pin_area, np.nan))), index=gdf.index)
_area_src = pd.Series(np.where(_override, 'pin_override',
                      np.where(_has_opa, 'opa_total_area',
                      np.where(_has_pin, 'pin_dor', 'knn'))), index=gdf.index)

# KNN-impute the remainder from neighbours already on this convention, so every parcel
# has an area before the LYCD step (Step 4's has_area is then universally true).
_need_area = _area.isna().values
n_area_knn = int(_need_area.sum())
if n_area_knn:
    _m = gdf.to_crs('EPSG:3857')
    _co = np.column_stack([_m.geometry.x, _m.geometry.y])
    _v = _area.values.astype(float)
    _have = ~np.isnan(_v)
    _tree = cKDTree(_co[_have])
    _, _ix = _tree.query(_co[_need_area], k=min(5, int(_have.sum())), workers=-1)
    _v[_need_area] = np.median(_v[_have][_ix], axis=1)
    _area = pd.Series(_v, index=gdf.index)

gdf['dor_area_sqft'] = _area
gdf['area_source'] = _area_src

print('\nLot area source:')
print(_area_src.value_counts().to_string())
print(f'  OPA records overridden by surveyed polygon: {int(_override.sum()):,}')

_PHILLY_SQFT = 134.2 * 27_878_400   # Philadelphia land area, 134.2 sq mi
_tot = gdf['dor_area_sqft'].sum()
print(f'\nTotal lot area: {_tot:.3e} sqft = {_tot/_PHILLY_SQFT:.2f}x the city '
      f'(expect <1.0 — parcels exclude streets and water)')
print(f'Median lot:     {gdf["dor_area_sqft"].median():,.0f} sqft')
assert _tot / _PHILLY_SQFT < 1.5, (
    f'Total lot area is {_tot/_PHILLY_SQFT:.2f}x the city — the area layer is inflated. '
    'Check that PIN areas are Mercator-corrected and that no spatial join has been reintroduced.'
)

# Join GMA zone labels
gdf = gdf.merge(
    gma[['key', 'gma1', 'gma2', 'gma3']],
    left_on='parcel_number', right_on='key',
    how='left',
)

gma_matched = gdf['gma3'].notna()
print(f'\nParcels with GMA assignment:      {gma_matched.sum():,} ({gma_matched.mean():.1%})')
print(f'Parcels needing GMA KNN fallback:  {(~gma_matched).sum():,}')

GMA assignment: 528,204 parcels | L1=17 zones | L2=84 | L3=613



Lot area source:
opa_total_area    550745
knn                30706
pin_dor             1360
pin_override         393
  OPA records overridden by surveyed polygon: 393

Total lot area: 3.147e+09 sqft = 0.84x the city (expect <1.0 — parcels exclude streets and water)
Median lot:     1,350 sqft



Parcels with GMA assignment:      527,496 (90.4%)
Parcels needing GMA KNN fallback:  55,708


## Step 4: Compute GMA hierarchical LYCD land values

For each parcel, land value is estimated using the "Least You Can Do" (LYCD) method
applied at OPA's Geographic Market Area (GMA) zone level:

1. For each GMA zone at the finest applicable level (L3 if ≥ 50 improved parcels,
   else L2, else L1): compute `median(market_value / dor_area_sqft)` over **improved**
   parcels in that zone. "Improved" = OPA category code not in {6, 12, 13}.
2. Apply land allocation: 20% for improved parcels (OPA's standard), 100% for vacant.
3. `lycd_land_value = zone_land_psf × parcel_dor_area_sqft`.

Using `market_value` from the same `assessments WHERE year=2024` pull as the billing
values ensures vintage consistency. The GMA hierarchy (L1: 17 zones, L2: 84 zones,
L3: 613 micro-zones) is OPA's own internal geography for setting assessments.

KNN fallback (5 nearest neighbors) is applied for parcels without a GMA assignment
or without a DOR lot area.

In [5]:
LAND_PCT_IMPROVED = 0.20   # OPA's standard land allocation for improved parcels
LAND_PCT_VACANT   = 1.00   # vacant parcels: market value is all land
MIN_IMPROVED      = 50     # minimum improved parcels per GMA zone before falling back

# Improved vs vacant flag using OPA category codes (same codes as the four-override system)
VACANT_CODES = {'6', '12', '13'}
cat_raw = gdf['category_code'].astype(str).str.strip()
has_market   = gdf['market_value'].notna() & (gdf['market_value'] > 0)
has_area     = gdf['dor_area_sqft'].notna() & (gdf['dor_area_sqft'] > 0)
is_improved  = ~cat_raw.isin(VACANT_CODES) & has_market & has_area
is_vacant_p  =  cat_raw.isin(VACANT_CODES) & has_market & has_area

# Per-parcel total-value PSF (used only to build zone medians for improved parcels)
gdf['_total_psf'] = np.where(
    is_improved | is_vacant_p,
    gdf['market_value'] / gdf['dor_area_sqft'],
    np.nan,
)

# Zone medians of total PSF over improved parcels at each GMA level
imp = gdf[is_improved]
l3_cnt = imp.groupby('gma3')['_total_psf'].count().rename('_l3_n')
l2_cnt = imp.groupby('gma2')['_total_psf'].count().rename('_l2_n')
l3_med = imp.groupby('gma3')['_total_psf'].median().rename('_l3_med')
l2_med = imp.groupby('gma2')['_total_psf'].median().rename('_l2_med')
l1_med = imp.groupby('gma1')['_total_psf'].median().rename('_l1_med')

for col_key, series in [('gma3', l3_cnt), ('gma3', l3_med),
                         ('gma2', l2_cnt), ('gma2', l2_med),
                         ('gma1', l1_med)]:
    gdf = gdf.merge(series.reset_index(), on=col_key, how='left')

# Hierarchical zone-median selection
gdf['_gma_med'] = np.where(
    gdf['_l3_n'].fillna(0) >= MIN_IMPROVED, gdf['_l3_med'],
    np.where(gdf['_l2_n'].fillna(0) >= MIN_IMPROVED, gdf['_l2_med'], gdf['_l1_med'])
)

# Apply land allocation per parcel type
_land_pct = np.where(is_vacant_p, LAND_PCT_VACANT, LAND_PCT_IMPROVED)
_land_psf  = gdf['_gma_med'] * _land_pct

# Land value for GMA-assigned parcels (every parcel already has an area from Step 3)
gdf['lycd_land_value'] = np.where(
    gdf['gma3'].notna() & has_area,
    (_land_psf * gdf['dor_area_sqft']).clip(lower=0),
    np.nan,
)

# Track GMA level used
gdf['gma_level'] = np.where(
    gdf['gma3'].isna(), 'knn',
    np.where(gdf['_l3_n'].fillna(0) >= MIN_IMPROVED, 'L3',
    np.where(gdf['_l2_n'].fillna(0) >= MIN_IMPROVED, 'L2', 'L1'))
)

n_gma       = int(gdf['lycd_land_value'].notna().sum())
n_knn_needed = int(gdf['lycd_land_value'].isna().sum())
print(f'GMA land values computed:      {n_gma:,}')
print(f'Parcels needing KNN fallback:  {n_knn_needed:,}')
print()

# Project to EPSG:3857 for KNN
gdf_m  = gdf.to_crs('EPSG:3857')
coords = np.column_stack([gdf_m.geometry.x, gdf_m.geometry.y])

def knn_impute(values, coords, K=5):
    values = np.asarray(values, dtype=float)
    has_v  = ~np.isnan(values)
    need_v = np.isnan(values)
    if need_v.sum() == 0:
        return values
    tree = cKDTree(coords[has_v])
    _, idxs = tree.query(coords[need_v], k=min(K, int(has_v.sum())), workers=-1)
    out = values.copy()
    out[need_v] = np.median(values[has_v][idxs], axis=1)
    return out

# Lot area is fully resolved in Step 3, so the only KNN needed here is for parcels
# outside GMA coverage. Note this imputes a dollar land VALUE from neighbours rather
# than a zone PSF, so an imputed parcel's own lot size does not enter its land value.
gdf['lycd_land_value'] = knn_impute(gdf['lycd_land_value'].values, coords)
print(f'KNN-imputed lycd_land_value for {n_knn_needed:,} parcels (no GMA zone)')

# Cap lycd_land_value at market_value for improved parcels (non-vacant OPA codes).
# Vacant parcels are exempt from the cap: OPA systematically undervalues them and
# the LYCD rate (100% zone PSF) reflects their development potential.
#
# Two distinct things make this bind, and they are worth keeping separate:
#   1. Genuinely large lots whose zone PSF overshoots their own value.
#   2. Zone-median overshoot on low-value homes — capped parcels have a median market
#      value around half that of uncapped ones, so the cap binds hardest on cheap houses
#      in expensive zones. This is a real distributional effect, not a data artifact.
# The shared-polygon area inflation that used to dominate this cap is gone as of the
# Step 2 rewrite; the remaining clip is the two mechanisms above.
_is_vac_cap = gdf['category_code'].astype(str).str.strip().isin({'6', '12', '13'})
_mv_cap = pd.to_numeric(gdf['market_value'], errors='coerce').fillna(0).clip(lower=0)
_pre_cap_total = float(gdf['lycd_land_value'].sum())
_capped_mask = (gdf['lycd_land_value'] > _mv_cap) & ~_is_vac_cap
_n_capped = int(_capped_mask.sum())
gdf['lycd_land_value'] = np.where(
    ~_is_vac_cap,
    np.minimum(gdf['lycd_land_value'], _mv_cap),
    gdf['lycd_land_value'],
)
_post_cap_total = float(gdf['lycd_land_value'].sum())
_removed = _pre_cap_total - _post_cap_total
print(f'\nMarket-value cap (improved parcels only):')
print(f'  parcels capped:      {_n_capped:,} ({_n_capped/len(gdf):.1%} of all)')
print(f'  land base pre-cap:   ${_pre_cap_total/1e9:.2f}B')
print(f'  land base post-cap:  ${_post_cap_total/1e9:.2f}B')
print(f'  removed by the cap:  ${_removed/1e9:.2f}B ({_removed/_pre_cap_total:.1%} of pre-cap base)')

# Clean up temporaries
gdf.drop(columns=['_total_psf', '_gma_med', '_l3_n', '_l2_n',
                   '_l3_med', '_l2_med', '_l1_med'], inplace=True)

print()
print('GMA level used:')
print(gdf['gma_level'].value_counts().to_string())
print()
print(f'All parcels have lycd_land_value: {gdf["lycd_land_value"].isna().sum() == 0}')

GMA land values computed:      527,496
Parcels needing KNN fallback:  55,708



KNN-imputed lycd_land_value for 55,708 parcels (no GMA zone)

Market-value cap (improved parcels only):
  parcels capped:      21,702 (3.7% of all)
  land base pre-cap:   $106.50B
  land base post-cap:  $82.46B
  removed by the cap:  $24.04B (22.6% of pre-cap base)

GMA level used:
gma_level
L3     526459
knn     55708
L2        931
L1        106

All parcels have lycd_land_value: True


## Step 5: Summarize LYCD land base

In [6]:
total_lycd_land = gdf['lycd_land_value'].sum()
total_opa_land  = gdf['taxable_land'].sum()
print(f'Total LYCD land base:    ${total_lycd_land/1e9:.2f}B')
print(f'Total OPA taxable land:  ${total_opa_land/1e9:.2f}B')
print(f'Ratio LYCD/OPA:          {total_lycd_land/total_opa_land:.2f}x')
print()
print('LYCD land value by GMA level (median $):')
print(gdf.groupby('gma_level')['lycd_land_value'].median().sort_values(ascending=False).to_string())
print()
print('LYCD land value percentiles (all parcels):')
for p in [10, 25, 50, 75, 90, 99]:
    v = gdf['lycd_land_value'].quantile(p/100)
    print(f'  p{p:2d}: ${v:,.0f}')


Total LYCD land base:    $82.46B
Total OPA taxable land:  $43.02B
Ratio LYCD/OPA:          1.92x

LYCD land value by GMA level (median $):
gma_level
L2     1.055800e+06
L1     6.270000e+05
knn    1.263000e+05
L3     4.594761e+04

LYCD land value percentiles (all parcels):
  p10: $20,646
  p25: $30,857
  p50: $49,091


  p75: $85,475
  p90: $173,238
  p99: $1,113,073


## Step 6: Categorize parcels (same overrides as OPA model)

In [7]:
gdf['category_code'] = (
    pd.to_numeric(gdf['category_code'], errors='coerce')
    .astype('Int64')
    .astype(str)
)

CATEGORY_MAP = {
    '1':  'Single Family Residential',
    '2':  'Small Multi-Family (2-4 units)',
    '3':  'Mixed Use',
    '4':  'Commercial',
    '5':  'Industrial',
    '6':  'Vacant Land',
    '7':  'Other Commercial',
    '8':  'Other Residential',
    '9':  'Hotel',
    '10': 'Office / Commercial Condo',
    '11': 'Other',
    '12': 'Vacant Land',
    '13': 'Vacant Land',
    '14': 'Large Multi-Family (5+ units)',
    '15': 'Retail / General Commercial',
}
gdf['PROPERTY_CATEGORY'] = gdf['category_code'].map(CATEGORY_MAP).fillna('Other')

# Override 1: $0 improvement -> Vacant Land
gdf.loc[gdf['taxable_building'] <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'

# Override 2: abated improved parcels
GENUINE_VACANT_CODES = {'6', '12', '13'}
abated_mask = (
    (gdf['taxable_building'] <= 0) &
    (gdf['taxable_land'] > 0) &
    (~gdf['category_code'].isin(GENUINE_VACANT_CODES))
)
gdf.loc[abated_mask, 'PROPERTY_CATEGORY'] = 'Abated / Construction Exemption'

# Override 3: OPA-vacant with nonzero building value
improved_vacant_mask = (
    gdf['category_code'].isin(GENUINE_VACANT_CODES) &
    (gdf['taxable_building'] > 0)
)
gdf.loc[improved_vacant_mask, 'PROPERTY_CATEGORY'] = 'Improved Vacant Land'

gdf['taxable_total'] = (gdf['taxable_land'] + gdf['taxable_building']).clip(lower=0)
gdf['full_exmp'] = (gdf['taxable_total'] <= 0).astype(int)

# Override 4: fully exempt parcels
EXEMPT_CATEGORY_MAP = {k: v + ' — Exempt' for k, v in CATEGORY_MAP.items()}
exempt_mask = gdf['full_exmp'] == 1
gdf.loc[exempt_mask, 'PROPERTY_CATEGORY'] = (
    gdf.loc[exempt_mask, 'category_code']
    .map(EXEMPT_CATEGORY_MAP)
    .fillna('Other — Exempt')
)

print(f'Total parcels: {len(gdf):,}')
print(f'Fully exempt: {gdf["full_exmp"].sum():,}  |  '
      f'Abated: {abated_mask.sum():,}  |  '
      f'Improved vacant: {improved_vacant_mask.sum():,}  |  '
      f'Taxable: {(gdf["full_exmp"] == 0).sum():,}')
print()
print('Property category distribution:')
print(gdf['PROPERTY_CATEGORY'].value_counts().to_string())

Total parcels: 583,204
Fully exempt: 36,938  |  Abated: 29,294  |  Improved vacant: 867  |  Taxable: 546,266

Property category distribution:
PROPERTY_CATEGORY
Single Family Residential                  416778
Small Multi-Family (2-4 units)              38610
Vacant Land                                 29530
Abated / Construction Exemption             29294
Single Family Residential — Exempt          20101
Mixed Use                                   13665
Vacant Land — Exempt                        11721
Commercial                                   8760
Industrial                                   3551
Commercial — Exempt                          3285
Large Multi-Family (5+ units)                2994
Other Residential                            1088
Small Multi-Family (2-4 units) — Exempt      1020
Improved Vacant Land                          867
Office / Commercial Condo                     827
Large Multi-Family (5+ units) — Exempt        259
Industrial — Exempt                     

## Step 7: Post-Abatement Baseline Tax

Pre-shift baseline treats all active construction abatements as **expired**: the building value
in `exempt_building` is restored to the taxable base for abated parcels before computing `current_tax`.
This produces a **higher revenue baseline than FY2024 actuals** — the LVT reform is revenue-neutral
relative to this higher counterfactual baseline.


In [8]:
gdf['millage_rate'] = MILLAGE

# Post-abatement baseline: restore abated building values.
gdf['model_building'] = pd.to_numeric(gdf['taxable_building'], errors='coerce').fillna(0).clip(lower=0)

abated = gdf['PROPERTY_CATEGORY'] == 'Abated / Construction Exemption'
exempt_bldg  = pd.to_numeric(gdf['exempt_building'], errors='coerce').fillna(0)
market_val   = pd.to_numeric(gdf['market_value'],    errors='coerce').fillna(0)
tax_land     = pd.to_numeric(gdf['taxable_land'],     errors='coerce').fillna(0)
implied_bldg = (market_val - tax_land).clip(lower=0)
abated_bldg  = exempt_bldg.where(exempt_bldg > 0, implied_bldg)
gdf.loc[abated, 'model_building'] = abated_bldg[abated].values

n_exempt_bldg = int((abated & (exempt_bldg > 0)).sum())
n_fallback    = int((abated & (exempt_bldg <= 0)).sum())
abated_bldg_total = gdf.loc[abated, 'model_building'].sum()
print(f'Abated parcels using exempt_building:        {n_exempt_bldg:,}')
print(f'Abated parcels using market_value fallback:  {n_fallback:,}')
print(f'Total abated building base restored:         ${abated_bldg_total/1e9:.2f}B')
print()

# post_abatement_total: OPA taxable_land + model_building (abatements treated as expired)
gdf['post_abatement_total'] = (
    pd.to_numeric(gdf['taxable_land'], errors='coerce').fillna(0).clip(lower=0)
    + gdf['model_building']
).clip(lower=0)

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='post_abatement_total',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

pre_abatement_levy = gdf['taxable_total'].mul(MILLAGE / 1000).sum()
print(f'Post-abatement baseline (city + school): ${current_revenue:,.0f}')
print(f'Pre-abatement combined levy (modeled):   ${pre_abatement_levy:,.0f}')
print(f'Added by restoring abated buildings:     ${current_revenue - pre_abatement_levy:,.0f}')


Abated parcels using exempt_building:        28,228
Abated parcels using market_value fallback:  1,066
Total abated building base restored:         $17.44B



Post-abatement baseline (city + school): $2,386,733,996
Pre-abatement combined levy (modeled):   $2,142,649,826
Added by restoring abated buildings:     $244,084,170


## Step 8: Build LYCD Reform Base (Post-Abatement)

Land value: GMA hierarchical LYCD (`lycd_land_value`).
Building value: `model_building` from Step 7 (restores `exempt_building` for abated parcels).
Revenue target: post-abatement baseline `current_tax` from Step 7.


In [9]:
gdf['model_land'] = gdf['lycd_land_value'].clip(lower=0)
# model_building already set in Step 7 (includes restored abated buildings).

print(f'Reform land base (LYCD):        ${gdf["model_land"].sum()/1e9:.2f}B')
print(f'Reform improvement base:         ${gdf["model_building"].sum()/1e9:.2f}B')
print(f'  of which abated bldg:          ${gdf.loc[abated, "model_building"].sum()/1e9:.2f}B')
print(f'OPA taxable land base:           ${pd.to_numeric(gdf["taxable_land"], errors="coerce").sum()/1e9:.2f}B')
print(f'OPA taxable building base:       ${pd.to_numeric(gdf["taxable_building"], errors="coerce").sum()/1e9:.2f}B')


Reform land base (LYCD):        $82.46B
Reform improvement base:         $127.48B
  of which abated bldg:          $17.44B
OPA taxable land base:           $43.02B
OPA taxable building base:       $110.05B


## Step 9: Revenue-neutral split-rate model (4:1 land:improvement)

In [10]:
taxable = gdf[gdf['full_exmp'] == 0].copy()

land_millage, improvement_millage, new_revenue, taxable = model_split_rate_tax(
    df=taxable,
    land_value_col='model_land',
    improvement_value_col='model_building',
    current_revenue=taxable['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

# Recombine exempt parcels
exempt = gdf[gdf['full_exmp'] == 1].copy()
exempt['new_tax'] = 0.0
exempt['tax_change'] = 0.0
exempt['tax_change_pct'] = 0.0
exempt['taxable_land_value'] = 0.0
exempt['taxable_improvement_value'] = 0.0
gdf = pd.concat([taxable, exempt]).sort_index()

print(f'Land millage:        {land_millage:.4f} mills')
print(f'Improvement millage: {improvement_millage:.4f} mills')
print(f'Revenue check:       ${new_revenue:,.0f} (target: ${taxable["current_tax"].sum():,.0f})')
print()

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title='Philadelphia — 4:1 Split-Rate Tax Impact (LYCD Land Values)')

Land millage:        25.2573 mills
Improvement millage: 6.3143 mills
Revenue check:       $2,386,733,996 (target: $2,386,733,996)




Philadelphia — 4:1 Split-Rate Tax Impact (LYCD Land Values)
                               Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
              Single Family Residential 416778    $-85,137,590       -6.9%      $-204        $-360   -1.6%     -17.3%            22.0%            60.2%
         Small Multi-Family (2-4 units)  38610    $-70,716,575      -28.0%    $-1,832      $-1,112  -21.3%     -28.4%             8.1%            82.1%
                            Vacant Land  29530    $333,951,326      744.4%    $11,309       $2,785 1296.7%     640.3%            97.3%             2.2%
        Abated / Construction Exemption  29294   $-112,274,091      -38.6%    $-3,833        $-425  -21.9%     -24.2%             7.2%            81.7%
     Single Family Residential — Exempt  20101              $0        0.0%         $0           $0    0.0%       0.0%             0.0%             0.0%
                           

## Step 10: Census join

In [11]:
import concurrent.futures

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f'Census join: {gdf["std_geoid"].notna().mean()*100:.1f}% matched')
        except concurrent.futures.TimeoutError:
            print('Census API timed out — skipping census join')
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f'Census join failed: {e}')
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

Census join: 100.0% matched


## Step 11: Export and visualize

In [12]:
out_df = save_standard_export(
    df=gdf,
    city=f'{CITY_NAME}{EXPORT_SUFFIX}',
    output_path=f'../../analysis/data/{CITY_NAME}{EXPORT_SUFFIX}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
    parcel_id_col='parcel_number',
)

from lvt.viz import create_city_report
create_city_report(out_df, f'{CITY_NAME}{EXPORT_SUFFIX}', show=False)
print('Done.')

  [warn] philadelphia_lycd_post_abatement_ty2026: non-standard property categories (will be preserved): ['Abated / Construction Exemption', 'Commercial — Exempt', 'Hotel — Exempt', 'Improved Vacant Land', 'Industrial — Exempt', 'Large Multi-Family (5+ units) — Exempt', 'Mixed Use — Exempt', 'Office / Commercial Condo — Exempt', 'Other Commercial — Exempt', 'Other Residential — Exempt', 'Other — Exempt', 'Retail / General Commercial — Exempt', 'Single Family Residential — Exempt', 'Small Multi-Family (2-4 units) — Exempt', 'Vacant Land — Exempt']


  ✓ philadelphia_lycd_post_abatement_ty2026: 583,204 rows → ../../analysis/data/philadelphia_lycd_post_abatement_ty2026.csv  [model: split_rate_4to1_lycd_post_abatement_ty2026]


create_city_report: excluded 36,938 fully-exempt parcels (583,204 → 546,266 modeled).


Done.
